# TeksMesin - Bahasa Kita ke Bahasa Mesin

Tugas individu: mengubah bahasa kita menjadi bahasa mesin.

Cara kerja singkatnya: tiap huruf punya kode angka, namanya kode ASCII.
Misalnya huruf A itu 65, terus 65 diubah ke biner jadi 01000001.
Jadi satu huruf biasa diwakili 8 digit biner.

Cara pakai notebook ini di Google Colab:
1. Upload file ini ke Colab, atau buka dari GitHub lewat Colab.
2. Jalankan semua sel dari atas ke bawah (Runtime, Run all).
3. Di sel interaktif paling bawah, tulis kalimat atau kode binernya,
   pilih arahnya, terus tekan tombol Ubah.

Catatan: versi aplikasi jendelanya (Tkinter) ada di file TeksMesin.py
dan versi compile-nya ada di TeksMesin.exe. Notebook ini isinya logika
konversi yang sama, dibuat khusus supaya bisa jalan di Colab yang
tidak punya tampilan jendela.


In [ ]:
def text_to_binary(text):
    """Teks -> biner, 8 digit per byte, dipisah spasi."""
    data = text.encode("utf-8")
    return " ".join("{:08b}".format(byte) for byte in data)


def binary_to_text(code):
    """Biner -> teks. Boleh pakai spasi atau disambung terus."""
    cleaned = "".join(code.split())
    if not cleaned:
        raise ValueError("Kode binernya masih kosong.")
    if any(ch not in "01" for ch in cleaned):
        raise ValueError("Kode biner hanya boleh berisi angka 0 dan 1.")
    if len(cleaned) % 8 != 0:
        raise ValueError(
            "Panjang kode biner harus kelipatan 8, karena satu huruf "
            "diwakili 8 digit. Sekarang panjangnya {} digit.".format(
                len(cleaned)))
    data = bytes(int(cleaned[i:i + 8], 2)
                 for i in range(0, len(cleaned), 8))
    try:
        return data.decode("utf-8")
    except UnicodeDecodeError:
        raise ValueError(
            "Byte-byte ini tidak membentuk huruf yang valid (bukan "
            "pengkodean UTF-8 yang benar).")


def byte_rows(data):
    """Satu baris per byte: (nomor, biner, desimal, karakter)."""
    rows = []
    for i, byte in enumerate(data, start=1):
        if byte == 32:
            char = "spasi"
        elif byte == 10:
            char = "enter"
        elif 33 <= byte <= 126:
            char = chr(byte)
        else:
            char = "-"
        rows.append((i, "{:08b}".format(byte), byte, char))
    return rows


In [ ]:
# Contoh cepat, tinggal jalankan sel ini.
kalimat = "Halo, selamat pagi!"
biner = text_to_binary(kalimat)
print("Teks :", kalimat)
print("Biner:", biner)
print()
print("Rincian per byte (kode ASCII):")
for nomor, b, desimal, karakter in byte_rows(kalimat.encode("utf-8")):
    print(f"byte {nomor}: {b}  = {desimal}  ({karakter})")
print()
print("Dibalikin lagi:", binary_to_text(biner))
print("Huruf A itu", 65, "binernya", text_to_binary("A"))


In [ ]:
# Sel interaktif. Ganti isi kolomnya, pilih arahnya, tekan Ubah.
try:
    import ipywidgets as widgets
    from IPython.display import display

    mode = widgets.Dropdown(
        options=[("Teks ke biner", "teks"), ("Biner ke teks", "biner")],
        value="teks",
        description="Arah:",
    )
    masukan = widgets.Textarea(
        value="Halo, selamat pagi!",
        description="Masukan:",
        layout=widgets.Layout(width="100%", height="80px"),
    )
    tombol = widgets.Button(description="Ubah", button_style="success")
    hasil = widgets.Output()

    def on_ubah(_):
        with hasil:
            hasil.clear_output()
            teks = masukan.value.strip()
            if not teks:
                print("Kotaknya masih kosong. Tulis dulu, baru diubah.")
                return
            try:
                if mode.value == "teks":
                    keluar = text_to_binary(teks)
                    data = teks.encode("utf-8")
                    print("Hasil (bahasa mesin):")
                    print(keluar)
                else:
                    keluar = binary_to_text(teks)
                    bersih = "".join(teks.split())
                    data = bytes(int(bersih[i:i + 8], 2)
                                 for i in range(0, len(bersih), 8))
                    print("Hasil (bahasa kita):")
                    print(keluar)
                print()
                print("Rincian per byte:")
                for nomor, b, desimal, karakter in byte_rows(data):
                    print(f"byte {nomor}: {b}  = {desimal}  ({karakter})")
            except ValueError as e:
                print("Gagal:", e)

    tombol.on_click(on_ubah)
    display(mode, masukan, tombol, hasil)
except ImportError:
    print("ipywidgets tidak tersedia di sini. Pakai cara manual di bawah:")
    print('teks = input("Tulis kalimat / kode biner: ")')
    print('print(text_to_binary(teks))  # untuk teks ke biner')
    print('print(binary_to_text(teks))  # untuk biner ke teks')
